In [1]:
import pandas as pd
import numpy as np
import torch
import strats
import datetime
import os
import pickle
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import ParameterGrid
from expl_perf_drop.explainers import CGExplainerDR
from expl_perf_drop.utils import Graph
from tqdm.notebook import tqdm
tqdm.pandas()

print(datetime.datetime.now())


Initializing package . . .
2025-02-10 20:32:58.565093


# Data processing

In [ ]:
def aggregate_time_series_features(df, obs_window=48*60, trim_percentile=0.01):
    """
    각 id, itemid 그룹별로 첫 'obs_window' 동안의 time series 데이터를 요약하는 함수.
    
    아래 feature들을 생성합니다.
      - mean: 평균값
      - max: 최댓값
      - min: 최솟값
      - std: 표준편차
      - median: 중앙값
      - q25: 25th percentile
      - q75: 75th percentile
      - range: 최대값 - 최소값
      - count: 관측치 개수
      - slope: offset(시간)에 따른 value의 선형 회귀 기울기 (추세)
    
    Parameters:
      df (pd.DataFrame): 원본 DataFrame, 컬럼은 ['hadm_id', 'offset', 'itemid', 'value']로 구성됨.
      
    Returns:
      pd.DataFrame: 각 id가 한 행(row)을 구성하며, itemid별 요약 통계량이 컬럼으로 확장된 DataFrame.
    """
    df_filtered = df[df['offset']<obs_window].copy()

    # 2. [Auto-detect Binary ItemIDs] - 각 itemid의 value가 {0}, {1} 또는 {0,1}인 경우 이진 변수로 판단
    binary_itemids = set()
    for item_id, group in df_filtered.groupby('itemid'):
        unique_vals = set(group['value'].unique())
        # NaN이 있을 수 있으므로 제거
        unique_vals.discard(np.nan)
        if unique_vals.issubset({0, 1}):
            binary_itemids.add(item_id)
    
    print(f"[Auto-detect Binary ItemIDs] Found {len(binary_itemids)} itemids:", binary_itemids)
    
    # 3. 비이진 변수에 대해서 trimming 수행 (예: 1% trimming)
    if trim_percentile is not None and trim_percentile > 0:
        clip_bounds = {}
        # (a) 각 itemid별 하위 및 상위 quantile 값을 계산 (비이진 변수에 한함)
        for item_id, group in df_filtered.groupby('itemid'):
            if item_id in binary_itemids:
                continue  # binary 변수는 trimming하지 않음
            lower = group['value'].quantile(trim_percentile)
            upper = group['value'].quantile(1 - trim_percentile)
            clip_bounds[item_id] = (lower, upper)
        
        # (b) 각 itemid 그룹에서 해당 범위 내의 데이터만 남김
        trimmed_groups = []
        for item_id, group in df_filtered.groupby('itemid'):
            if item_id in binary_itemids:
                trimmed_groups.append(group)
            else:
                if item_id in clip_bounds:
                    l, u = clip_bounds[item_id]
                    trimmed = group[(group['value'] >= l) & (group['value'] <= u)]
                    trimmed_groups.append(trimmed)
                else:
                    # 드물게 train에 나타나지 않는 itemid: 그대로 둠
                    trimmed_groups.append(group)
        
        df_filtered = pd.concat(trimmed_groups, ignore_index=True)
        print(f"[Trimming] Applied {trim_percentile*100:.1f}% trimming for non-binary itemids.")
    else:
        print("[Trimming] Skipped or trim_percentile=0")
    
    # 2. 사용자 정의 aggregation 함수
    def agg_funcs(group):
        res = {}
        values = group['value']
        res['mean'] = values.mean()
        res['max'] = values.max()
        res['min'] = values.min()
        res['std'] = values.std()
        res['median'] = values.median()
        res['q25'] = values.quantile(0.25)
        res['q75'] = values.quantile(0.75)
        res['range'] = values.max() - values.min()
        res['count'] = values.count()
        return pd.Series(res)
    
    # 3. id와 itemid별로 aggregation 적용 (groupby 후 사용자 정의 함수 적용)
    agg_df = df_filtered.groupby(['hadm_id', 'itemid'])[['hadm_id', 'offset','itemid', 'value']].progress_apply(agg_funcs)
    
    # 4. 각 id가 한 행이 되도록 피벗 (itemid가 컬럼이 됨)
    pivot_df = agg_df.unstack('itemid')
    
    # 5. MultiIndex 형태의 컬럼명을 flatten: 'itemid_stat' 형식으로 변경
    pivot_df.columns = [f'{item}_{stat}' for stat, item in pivot_df.columns]
    
    # 6. id를 컬럼으로 변환
    pivot_df = pivot_df.reset_index()
    
    return pivot_df


def compute_trimmed_stats(values: pd.Series, low_pct: float, high_pct: float):
    """
    values: 해당 그룹의 value Series
    low_pct, high_pct: 잘라낼 분위수 (예: 0.01, 0.99)
    
    반환: (trimmed_mean, trimmed_std)
    """
    lower_bound = values.quantile(low_pct)
    upper_bound = values.quantile(high_pct)
    trimmed = values[(values >= lower_bound) & (values <= upper_bound)]
    return trimmed.mean(), trimmed.std()

def calculate_all_stats(df: pd.DataFrame):
    # 결과를 담을 리스트
    results = []

    # itemid별로 그룹화
    grouped = df.groupby('itemid')

    for item_id, group in grouped:
        vals = group['value']

        # 1) 전체(아웃라이어 제거 없음) 평균/표준편차
        orig_mean = vals.mean()
        orig_std  = vals.std()

        # 2) 상하위 1% 제거
        mean_1pct, std_1pct = compute_trimmed_stats(vals, 0.01, 0.99)

        # 3) 상하위 3% 제거
        mean_3pct, std_3pct = compute_trimmed_stats(vals, 0.03, 0.97)

        # 4) 상하위 5% 제거
        mean_5pct, std_5pct = compute_trimmed_stats(vals, 0.05, 0.95)

        # 결과 한 줄로 정리
        results.append({
            'itemid': item_id,
            'orig_mean': orig_mean,
            'orig_std': orig_std,
            'mean_1pct': mean_1pct,
            'std_1pct': std_1pct,
            'mean_3pct': mean_3pct,
            'std_3pct': std_3pct,
            'mean_5pct': mean_5pct,
            'std_5pct': std_5pct
        })

    # 리스트를 DataFrame으로
    df_stats = pd.DataFrame(results)
    return df_stats


def aggregate_time_series_features_cf(df, 
                                            obs_window=24*60, 
                                            step=8*60,
                                            trim_percentile=0.01):
    """
    각 hadm_id에 대해, query_time을 0부터 max_offset까지 step 간격으로 순회.
    각 window = [query_time, query_time + obs_window)의 시계열 데이터를 요약.
    
    - 첫 열은 (hadm_id, query_time),
    - 이후 itemid 별로 (mean, max, min, std, median, q25, q75, range, count, slope) 등등
      -> 열이 확장되어 한 행(row)에 들어감.
      
    Parameters
    ----------
    df : pd.DataFrame
        columns = ['hadm_id','offset','itemid','value']
    obs_window : int
        window 길이(분). 기본 48시간(2880분)
    step : int
        query_time 간격(분). 기본 60분(1시간)
    trim_percentile : float
        예: 0.01 => 상하위 1% trimming. (0~1사이, None 또는 0 이하이면 skip)
        
    Returns
    -------
    pd.DataFrame
        각 (hadm_id, query_time) × itemid별 요약 통계가 열로 배치된 DataFrame
    """
    import pandas as pd
    import numpy as np
    from tqdm import tqdm

    # 1) 우선 obs_window 내에서만 쓰는 것은 아님. step windows를 사용할 것이므로 df 전체를 사용
    df_filtered = df.copy()

    # 2) 이진 변수 식별
    binary_itemids = set()
    for item_id, group in df_filtered.groupby('itemid'):
        unique_vals = set(group['value'].dropna().unique())
        if unique_vals.issubset({0,1}):
            binary_itemids.add(item_id)
    print(f"[Auto-detect Binary ItemIDs] Found {len(binary_itemids)} itemids:", binary_itemids)

    # 3) 비이진 변수 trimming
    if trim_percentile is not None and 0< trim_percentile <1:
        clip_bounds = {}
        for item_id, group in df_filtered.groupby('itemid'):
            if item_id in binary_itemids:
                continue
            lower = group['value'].quantile(trim_percentile)
            upper = group['value'].quantile(1 - trim_percentile)
            clip_bounds[item_id] = (lower, upper)

        trimmed_parts = []
        for item_id, grp in df_filtered.groupby('itemid'):
            if item_id in binary_itemids:
                trimmed_parts.append(grp)
            else:
                if item_id in clip_bounds:
                    l,u = clip_bounds[item_id]
                    clipped = grp[(grp['value']>=l)&(grp['value']<=u)]
                    trimmed_parts.append(clipped)
                else:
                    trimmed_parts.append(grp)
        df_filtered = pd.concat(trimmed_parts, ignore_index=True)
        print(f"[Trimming] Applied {trim_percentile*100:.1f}% trimming for non-binary itemids.")
    else:
        print("[Trimming] Skipped or trim_percentile=0")

    # 4) 통계 함수
    def agg_funcs(values):
        res = {}
        if len(values)==0:
            # 비어있으면 NaN
            return {
                'mean': np.nan, 'max': np.nan, 'min': np.nan, 'std': np.nan,
                'median': np.nan, 'q25': np.nan, 'q75': np.nan,
                'range': np.nan, 'count': 0,
                'slope': np.nan
            }
        res['mean'] = values.mean()
        res['max']  = values.max()
        res['min']  = values.min()
        res['std']  = values.std()
        res['median'] = np.median(values)
        res['q25'] = np.quantile(values, 0.25)
        res['q75'] = np.quantile(values, 0.75)
        res['range'] = res['max'] - res['min']
        res['count'] = len(values)
        # slope => offset vs value 회귀 기울기
        # 만약 offset이 필요하면 values array만으론 부족 -> 별도 인자로 받아야 함
        # 여기서는 제외하거나, 아래에서 따로 계산
        res['slope'] = np.nan  # 일단 NaN 처리 or 나중에 구현
        return res

    # 5) hadm_id별로 max offset 확인, query_time을 0 ~ max_offset까지 step
    #    각 window [query_time, query_time+obs_window) => itemid별 agg
    results = []  # 각 row를 저장할 list of dict

    for hadm_id, hadm_group in tqdm(df_filtered.groupby('hadm_id'), desc="[aggregate stepwise]"):
        if hadm_group.empty:
            continue
        max_offset = hadm_group['offset'].max()
        # step별 qtime
        qtimes = range(0, int(max_offset)+1, step)

        # 효율 위해, hadm_group를 itemid별로 미리 groupby 해둘 수도 있음 (여기선 간단하게)
        for qtime in qtimes:
            wstart = qtime
            wend   = qtime + obs_window
            window_df = hadm_group[(hadm_group['offset']>=wstart) & (hadm_group['offset']<wend)]
            if window_df.empty:
                continue

            # itemid별로 요약
            row_dict = {
                'hadm_id': hadm_id,
                'query_time': qtime
            }

            # groupby itemid => agg_funcs
            for item_id, subgrp in window_df.groupby('itemid'):
                vals = subgrp['value'].values
                # offset_array = subgrp['offset'].values # slope 등에 필요하면
                stats = agg_funcs(vals)
                for stat_name, stat_val in stats.items():
                    # 컬럼명: f'{item_id}_{stat_name}'
                    col_name = f'{item_id}_{stat_name}'
                    row_dict[col_name] = stat_val
            
            results.append(row_dict)

    # 6) 결과 DataFrame
    out_df = pd.DataFrame(results)
    # hadm_id, query_time => 첫 열, 그 후 itemid_stat 열들
    # (이미 dict key 순서가 (hadm_id, query_time) -> others 인지는 확실치 않으니, 정렬)
    base_cols = ['hadm_id','query_time']
    other_cols = [c for c in out_df.columns if c not in base_cols]
    final_cols = base_cols + other_cols
    out_df = out_df[final_cols]

    return out_df

In [ ]:
mimic_static = pd.read_feather('mimic_data_static.feather')

def process_static(static):
    # Treat only in-hospital death
    static.loc[static['hospital_expire_flag'] == 0, 'deathtime'] = None
    print(len(static[static['outtime'] >= static['deathtime']]))
    static = static.loc[:,['hadm_id', 'deathtime']]
    static = static.rename(columns={'hadm_id': 'hadm_id', 'deathtime' : 'death_offset'})
    unique_static_ids = static['hadm_id'].unique()
    return static, unique_static_ids

mimic_outcome, mimic_ids = process_static(mimic_static)

data_vital = pd.read_feather('mimic_data_vital.feather').rename(columns={'valuenum' : 'value'})
data_lab = pd.read_feather('mimic_data_lab.feather').rename(columns={'valuenum' : 'value'})
data_treatment = pd.read_feather('mimic_data_treatment.feather').rename(columns={'valuenum' : 'value'})

data = pd.concat([data_vital, data_lab, data_treatment])

# Factorize the 'item' column and get the mapping
encoded_total, actual_class_total = pd.factorize(data['itemid'])
# data.loc[:,'itemid'] = encoded_total.astype(int)
# data['itemid'] = pd.to_numeric(data['itemid'], errors='coerce')

data = data.dropna(subset=['value'])
data_stats = calculate_all_stats(data)

data_agg = aggregate_time_series_features_cf(data)

item_dict_total = dict(zip(actual_class_total, range(len(actual_class_total))))
emb_idx_total = len(item_dict_total)

# 재현성 유지를 위해 seed를 고정하고 섞습니다.
np.random.seed(9871)
np.random.shuffle(mimic_ids)

# 예: 80%를 train, 나머지를 valid로 사용
train_size = int(len(mimic_ids) * 0.6)
valid_size = int(len(mimic_ids) * 0.8)
train_hadm_ids = mimic_ids[:train_size]
valid_hadm_ids = mimic_ids[train_size:valid_size]
test_hadm_ids = mimic_ids[valid_size:]

train_final_ids = train_hadm_ids
valid_final_ids = valid_hadm_ids
test_final_ids = test_hadm_ids

# # (A) death_offset이 존재하는 hadm_id 목록
# hadm_with_death_offset = set(
#     mimic_outcome.loc[~mimic_outcome['death_offset'].isna(), 'hadm_id']
# )

# # (B) Train 아이디 중, death_offset 존재 vs 미존재
# train_hadm_set = set(train_hadm_ids)
# train_hadm_with_death = train_hadm_set & hadm_with_death_offset
# train_hadm_no_death = list(train_hadm_set - hadm_with_death_offset)

# # (C) no_death 중 40%만 샘플링
# sample_count = int(len(train_hadm_no_death) * 0.4)
# np.random.shuffle(train_hadm_no_death)

# train_hadm_no_death_sampled = train_hadm_no_death[:sample_count]

# # 최종 Train 아이디
# train_final_ids = set(train_hadm_with_death) | set(train_hadm_no_death_sampled)

# # Same for validation
# valid_hadm_set = set(valid_hadm_ids)
# valid_hadm_with_death = valid_hadm_set & hadm_with_death_offset
# valid_hadm_no_death = list(valid_hadm_set - hadm_with_death_offset)

# sample_count = int(len(valid_hadm_no_death) * 0.4)
# np.random.shuffle(valid_hadm_no_death)

# valid_hadm_no_death_sampled = valid_hadm_no_death[:sample_count]
# valid_final_ids = set(valid_hadm_with_death) | set(valid_hadm_no_death_sampled)

# # Also for test...
# test_hadm_set = set(test_hadm_ids)
# test_hadm_with_death = test_hadm_set & hadm_with_death_offset
# test_hadm_no_death = list(test_hadm_set - hadm_with_death_offset)

# sample_count = int(len(test_hadm_no_death) * 0.4)
# np.random.shuffle(test_hadm_no_death)

# test_hadm_no_death_sampled = test_hadm_no_death[:sample_count]
# test_final_ids = set(test_hadm_with_death) | set(test_hadm_no_death_sampled)

# (D) 최종 DF
train_df = data_agg[data_agg['hadm_id'].isin(train_final_ids)].copy().sort_values(by=['hadm_id']).drop(columns=['hadm_id']).reset_index(drop=True)
valid_df = data_agg[data_agg['hadm_id'].isin(valid_final_ids)].copy().sort_values(by=['hadm_id']).drop(columns=['hadm_id']).reset_index(drop=True)
test_df = data_agg[data_agg['hadm_id'].isin(test_final_ids)].copy().sort_values(by=['hadm_id']).drop(columns=['hadm_id']).reset_index(drop=True)

# train_features_df = aggregate_time_series_features(train_df).sort_values(by=['hadm_id'])
# valid_features_df = aggregate_time_series_features(valid_df).sort_values(by=['hadm_id'])
# test_features_df = aggregate_time_series_features(test_df).sort_values(by=['hadm_id'])

train_outcome_df = mimic_outcome[mimic_outcome['hadm_id'].isin(train_final_ids)].copy().sort_values(by=['hadm_id']).drop(columns=['hadm_id'])
valid_outcome_df = mimic_outcome[mimic_outcome['hadm_id'].isin(valid_final_ids)].copy().sort_values(by=['hadm_id']).drop(columns=['hadm_id'])
test_outcome_df = mimic_outcome[mimic_outcome['hadm_id'].isin(test_final_ids)].copy().sort_values(by=['hadm_id']).drop(columns=['hadm_id'])

train_outcome = train_outcome_df['death_offset'].notna()
valid_outcome = valid_outcome_df['death_offset'].notna()
test_outcome = test_outcome_df['death_offset'].notna()

train_df['outcome'] = train_outcome.values
valid_df['outcome'] = valid_outcome.values
test_df['outcome'] = test_outcome.values


print(datetime.datetime.now())

21
[Auto-detect Binary ItemIDs] Found 14 itemids: {'h', 'n', 'fluid', 'c_else', 'r', 'ventilator', 'l', 'a10', 'b', 'c01', 'a_supplements', 'v', 'a_drug', 'm'}
[Trimming] Applied 1.0% trimming for non-binary itemids.


  0%|          | 0/938132 [00:00<?, ?it/s]

2025-02-10 20:46:04.134033


In [4]:
train_emb_df = pd.read_csv('./model_results/train_embedding_mimic_initial_48_fusion_full_focal0515_64emb.csv').sort_values(by=['hadm_id']).drop(columns=['hadm_id', 'query_time']).reset_index(drop=True)
valid_emb_df = pd.read_csv('./model_results/valid_embedding_mimic_initial_48_fusion_full_focal0515_64emb.csv').sort_values(by=['hadm_id']).drop(columns=['hadm_id', 'query_time']).reset_index(drop=True)
test_emb_df = pd.read_csv('./model_results/test_embedding_mimic_initial_48_fusion_full_focal0515_64emb.csv').sort_values(by=['hadm_id']).drop(columns=['hadm_id', 'query_time']).reset_index(drop=True)

train_emb_df['outcome'] = train_outcome.values
valid_emb_df['outcome'] = valid_outcome.values
test_emb_df['outcome'] = test_outcome.values

In [5]:
# embs_varwise

train_vital_df = pd.read_csv('./model_results/train_embedding_mimic_vital_48_fusion_full_focal0515.csv').sort_values(by=['hadm_id']).drop(columns=['query_time'])
train_lab_df = pd.read_csv('./model_results/train_embedding_mimic_lab_48_fusion_full_focal0515.csv').sort_values(by=['hadm_id']).drop(columns=['query_time'])
train_treat_df = pd.read_csv('./model_results/train_embedding_mimic_treatment_48_fusion_full_focal0515.csv').sort_values(by=['hadm_id']).drop(columns=['query_time'])

train_vars_df = pd.merge(train_vital_df, train_lab_df, on='hadm_id', how='outer', suffixes=('_vital', '_lab'))
train_vars_df = pd.merge(train_vars_df, train_treat_df, on='hadm_id', how='outer').sort_values(by=['hadm_id']).drop(columns=['hadm_id']).reset_index(drop=True)

valid_vital_df = pd.read_csv('./model_results/valid_embedding_mimic_vital_48_fusion_full_focal0515.csv').sort_values(by=['hadm_id']).drop(columns=['query_time'])
valid_lab_df = pd.read_csv('./model_results/valid_embedding_mimic_lab_48_fusion_full_focal0515.csv').sort_values(by=['hadm_id']).drop(columns=['query_time'])
valid_treat_df = pd.read_csv('./model_results/valid_embedding_mimic_treatment_48_fusion_full_focal0515.csv').sort_values(by=['hadm_id']).drop(columns=['query_time'])

valid_vars_df = pd.merge(valid_vital_df, valid_lab_df, on='hadm_id', how='outer', suffixes=('_vital', '_lab'))
valid_vars_df = pd.merge(valid_vars_df, valid_treat_df, on='hadm_id', how='outer').sort_values(by=['hadm_id']).drop(columns=['hadm_id']).reset_index(drop=True)

test_vital_df = pd.read_csv('./model_results/test_embedding_mimic_vital_48_fusion_full_focal0515.csv').sort_values(by=['hadm_id']).drop(columns=['query_time'])
test_lab_df = pd.read_csv('./model_results/test_embedding_mimic_lab_48_fusion_full_focal0515.csv').sort_values(by=['hadm_id']).drop(columns=['query_time'])
test_treat_df = pd.read_csv('./model_results/test_embedding_mimic_treatment_48_fusion_full_focal0515.csv').sort_values(by=['hadm_id']).drop(columns=['query_time'])

test_vars_df = pd.merge(test_vital_df, test_lab_df, on='hadm_id', how='outer', suffixes=('_vital', '_lab'))
test_vars_df = pd.merge(test_vars_df, test_treat_df, on='hadm_id', how='outer').sort_values(by=['hadm_id']).drop(columns=['hadm_id']).reset_index(drop=True)

train_vars_df['outcome'] = train_outcome.values
valid_vars_df['outcome'] = valid_outcome.values
test_vars_df['outcome'] = test_outcome.values

# Representation modules

# Downstream modules

Input of the downstream modules for stay 'i' should be in form [rep1, rep2, ... , i_outcome]

In [ ]:
def train_and_evaluate_downstream_model(train, 
                                        valid,
                                        test,
                                          model_name='downstream_model',
                                          save_path=None,
                                          verbose=True):
    """
    downstream 모델을 학습, 평가하고 저장하는 함수.
    
    학습은 train 데이터와 valid 데이터를 사용하여 수행하며,
    최종적으로는 test 데이터에 대한 성능을 평가합니다.
    
    Parameters:
      - x_train, y_train: 학습 데이터의 representation 및 타겟 레이블.
      - x_valid, y_valid: validation 데이터의 representation 및 타겟 레이블 (fit 시 eval_set으로 사용).
      - x_test, y_test: test 데이터의 representation 및 타겟 레이블 (최종 성능 평가에 사용).
      - model_name: 저장 시 모델 파일 이름에 사용할 이름 (기본값 'downstream_model')
      - save_path: 저장할 디렉토리 또는 파일 경로 (None이면 저장하지 않음)
      - verbose: 학습 및 평가 결과 출력 여부
      
    Returns:
      - model: 학습된 XGBClassifier 모델.
      - metrics: test 데이터에 대한 평가 지표가 담긴 dict (AUROC, AUPRC, Precision, Recall, F1, Accuracy)
    """
    # 1. 모델 초기화 및 학습 (GPU 사용)
    x_train = train.drop(columns=['outcome'])
    y_train = train['outcome']
    x_valid = valid.drop(columns=['outcome'])
    y_valid = valid['outcome']    
    x_test = test.drop(columns=['outcome'])
    y_test = test['outcome']


    model = xgb.XGBClassifier(tree_method='hist', device='cuda')
    model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)

    # 2. test 데이터에 대한 예측 수행
    # AUROC, AUPRC 계산을 위해 양성 클래스 확률 사용
    y_pred_proba = model.predict_proba(x_test)[:, 1]
    # 그 외 지표는 이진 예측값 사용
    y_pred = model.predict(x_test)

    # 3. 평가 지표 계산 (test 데이터 기준)
    metrics = {
        'AUROC': roc_auc_score(y_test, y_pred_proba),
        'AUPRC': average_precision_score(y_test, y_pred_proba),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred)
    }

    # 4. 평가 결과 출력
    if verbose:
        print(f"Metrics for {model_name} on Test Set:")
        for key, value in metrics.items():
            print(f"{key}: {value:.4f}")

    # 5. 모델 저장 (save_path가 제공된 경우)
    if save_path is not None:
        # save_path가 디렉토리인 경우 파일명을 추가
        if os.path.isdir(save_path):
            file_path = os.path.join(save_path, f"{model_name}.pkl")
        else:
            file_path = save_path

        with open(file_path, 'wb') as f:
            pickle.dump(model, f)
        if verbose:
            print(f"Model saved to {file_path}")

    return model, metrics

def grid_search_train_and_evaluate_downstream_model(train, 
                                                      valid,
                                                      test,
                                                      param_grid,
                                                      model_name='downstream_model',
                                                      save_path=None,
                                                      verbose=True):
    """
    grid search로 downstream 모델의 하이퍼파라미터를 탐색하여, 
    validation set의 AUPRC가 가장 높은 모델을 선택하고 test set 성능을 평가하는 함수.
    
    Parameters:
      - train, valid, test: 각 데이터셋은 'outcome' 열을 포함한 DataFrame.
          'outcome' 열은 타겟 레이블(예: 이진 변수)을 의미하며,
          나머지 컬럼들은 모델의 feature로 사용됩니다.
      - param_grid: 탐색할 하이퍼파라미터의 dictionary (예: {'max_depth': [3, 5], 'learning_rate': [0.01, 0.1]})
      - model_name: 저장 시 모델 파일 이름에 사용할 이름 (기본값 'downstream_model')
      - save_path: 모델 저장 경로 (디렉토리 또는 파일 경로). None이면 저장하지 않음.
      - verbose: 학습 및 평가 결과 출력 여부
      
    Returns:
      - best_model: 선택된 XGBClassifier 모델.
      - best_params: best_model의 하이퍼파라미터 dict.
      - best_valid_auprc: validation set에서 측정한 AUPRC 값.
      - test_metrics: test set에 대한 평가 지표가 담긴 dict (AUROC, AUPRC, Precision, Recall, F1, Accuracy)
    """
    # 데이터 분리 (feature와 target)
    x_train = train.drop(columns=['outcome'])
    y_train = train['outcome']
    x_valid = valid.drop(columns=['outcome'])
    y_valid = valid['outcome']
    x_test = test.drop(columns=['outcome'])
    y_test = test['outcome']
    
    best_valid_auprc = -np.inf
    best_params = None
    best_model = None

    # grid search 수행: param_grid의 모든 조합에 대해 모델 학습 및 validation 평가
    for params in ParameterGrid(param_grid):
        # if verbose:
        #     print("Trying parameters:", params)
        # 모델 초기화 (추가 파라미터 전달, GPU 사용)
        model = xgb.XGBClassifier(tree_method='hist', device='cuda', **params)
        model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)
        
        # validation set에 대한 예측 수행
        valid_y_pred_proba = model.predict_proba(x_valid)[:, 1]
        valid_auprc = average_precision_score(y_valid, valid_y_pred_proba)
        # if verbose:
        #     print("Validation AUPRC: {:.4f}".format(valid_auprc))
        
        # 가장 좋은 AUPRC를 가진 모델 선택
        if valid_auprc > best_valid_auprc:
            best_valid_auprc = valid_auprc
            best_params = params
            best_model = model

    # best_model을 사용하여 test set 평가
    y_pred_proba = best_model.predict_proba(x_test)[:, 1]
    y_pred = best_model.predict(x_test)
    test_metrics = {
        'AUROC': roc_auc_score(y_test, y_pred_proba),
        'AUPRC': average_precision_score(y_test, y_pred_proba),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred)
    }
    
    if verbose:
        print("\nBest Hyperparameters:", best_params)
        print("Best Validation AUPRC: {:.4f}".format(best_valid_auprc))
        print("\nTest Metrics for the Best Model:")
        for key, value in test_metrics.items():
            print(f"{key}: {value:.4f}")

    # 모델 저장 (save_path가 제공된 경우)
    if save_path is not None:
        if os.path.isdir(save_path):
            file_path = os.path.join(save_path, f"{model_name}.pkl")
        else:
            file_path = save_path

        with open(file_path, 'wb') as f:
            pickle.dump(best_model, f)
        if verbose:
            print(f"Model saved to {file_path}")

    return best_model, best_params, best_valid_auprc, test_metrics

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.3],
    'n_estimators': [50, 100, 200],
    'subsample': [0.8, 1.0],
    'scale_pos_weight' : [1, 10, 24]
}

In [14]:
# Evaluate downstream model on REP_1
model_1, params_1, auprc_1, metrics_1 = grid_search_train_and_evaluate_downstream_model(
    train_emb_df,
    valid_emb_df,
    test_emb_df,
    param_grid,
    model_name='total_embs',
    save_path='./saved_models',
    verbose=True
)

# On REP_2 . . .
model_2, params_2, auprc_2, metrics_2 = grid_search_train_and_evaluate_downstream_model(
    train_df,
    valid_df,
    test_df,
    param_grid,
    model_name='ts_stats',
    save_path='./saved_models',
    verbose=True
)

model_3, params_3, auprc_3, metrics_3 = grid_search_train_and_evaluate_downstream_model(
    train_vars_df,
    valid_vars_df,
    test_vars_df,
    param_grid,
    model_name='varwise_embs',
    save_path='./saved_models',
    verbose=True
)

# Compare metrics...
print("\nComparison of Metrics:")
print("Representation 1:", metrics_1)
print("Representation 2:", metrics_2)
print("Representation 3:", metrics_3)


Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'scale_pos_weight': 24, 'subsample': 0.8}
Best Validation AUPRC: 0.2376

Test Metrics for the Best Model:
AUROC: 0.8531
AUPRC: 0.2446
Precision: 0.1216
Recall: 0.7109
F1: 0.2077
Accuracy: 0.7969
Model saved to ./saved_models

Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'scale_pos_weight': 24, 'subsample': 0.8}
Best Validation AUPRC: 0.2471

Test Metrics for the Best Model:
AUROC: 0.8623
AUPRC: 0.2317
Precision: 0.1534
Recall: 0.6460
F1: 0.2479
Accuracy: 0.8532
Model saved to ./saved_models

Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 50, 'scale_pos_weight': 24, 'subsample': 0.8}
Best Validation AUPRC: 0.2558

Test Metrics for the Best Model:
AUROC: 0.8432
AUPRC: 0.2229
Precision: 0.1488
Recall: 0.5870
F1: 0.2375
Accuracy: 0.8588
Model saved to ./saved_models

Comparison of Metrics:
Representation 1: {'AUROC': np.float64(0.8530504656637

# Explain

In [8]:
var_categories = { 
    'lab': ['G'],
    'treat': ['X1'],
    'vital': ['X2'],
    'outcome': ['Outcome']
}

# define the causal graph
GRAPH = Graph(
    nodes= list(var_categories.keys()),
    edges=[
        ('vital', 'lab'),
        ('vital', 'treat'),
        ('lab', 'treat'),
        ('vital', 'outcome'), 
        ('lab', 'outcome'), 
        ('treat', 'outcome'), 
    ]
)

In [9]:
def auprc(model, data, subset_cols=None, weight=None, target_name='Y'):
    """
    모델의 예측 확률을 사용해 AUPRC (Average Precision Score)를 계산하는 함수.

    Parameters:
      - model: 예측 모델 (predict_proba 메서드가 있어야 함)
      - data: 평가할 DataFrame
      - subset_cols: 사용할 feature 컬럼 리스트 (None이면 전체 데이터 사용)
      - weight: 샘플 가중치 (None이면 가중치 없이 계산)
      - target_name: 타겟 컬럼 이름 (기본값 'Y')

    Returns:
      - AUPRC 값 (float)
    """
    data_input = data[subset_cols] if subset_cols is not None else data
    return average_precision_score(data[target_name], model.predict_proba(data_input)[:, 1], sample_weight=weight)

In [ ]:
# exp = CGExplainerDR(GRAPH, train_vars_df, source_eval_df, target_train_df, target_eval_df,
#         TRAIN_FEATURES, VAR_CATEGORIES, target_name = 'Y'
# )

NameError: name 'source_eval_df' is not defined